# 01 - Data loading & cleaning

**Goal:** load all three project datasets, audit them, hit every known data landmine
on purpose, fix each one *with a documented reason*, and save clean processed copies.

In [24]:
import sys
from pathlib import Path

# Make src/ importable whether Jupyter starts in the project root or in Code/
for candidate in (Path.cwd(), Path.cwd() / "Code", Path.cwd().parent):
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate))
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)

## 1. IBM Telco - load the raw file

In [25]:
import zipfile

with zipfile.ZipFile("../Dataset/IBM_telco.zip") as zf:
    print(zf.namelist())
    with zf.open("WA_Fn-UseC_-Telco-Customer-Churn.csv") as f:
        telco_raw = pd.read_csv(f)

print(telco_raw.shape)
telco_raw.head()

['WA_Fn-UseC_-Telco-Customer-Churn.csv']
(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### 1.1 Structural audit

In [26]:
summary = pd.DataFrame({
    "dtype": telco_raw.dtypes.astype(str),
    "nulls": telco_raw.isna().sum(),
    "unique": telco_raw.nunique(),
})
print("duplicate customerIDs:", telco_raw["customerID"].duplicated().sum())
print("duplicate rows:", telco_raw.duplicated().sum())
summary

duplicate customerIDs: 0
duplicate rows: 0


,dtype,nulls,unique
customerID,object,0,7043
gender,object,0,2
SeniorCitizen,int64,0,2
Partner,object,0,2
Dependents,object,0,2
tenure,int64,0,73
PhoneService,object,0,2
MultipleLines,object,0,3
InternetService,object,0,3
OnlineSecurity,object,0,3


### 1.2 Landmine #1 - `TotalCharges` is text

In [27]:
bad = telco_raw[pd.to_numeric(telco_raw["TotalCharges"], errors="coerce").isna()]
print(f"{len(bad)} rows fail to parse")
print("their raw value:", repr(bad["TotalCharges"].unique()))
print("their tenure values:", bad["tenure"].unique())
bad[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

11 rows fail to parse
their raw value: array([' '], dtype=object)
their tenure values: [0]


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


All 11 broken rows are a **single space** - and all have `tenure = 0`. These are
brand-new customers who have never been billed. So the faithful fix is
**impute 0**, not drop: dropping them would teach the model that brand-new
customers don't exist, and the pipeline will meet plenty of them in production.

In [28]:
telco = telco_raw.copy()
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0.0)
assert telco["TotalCharges"].dtype == "float64" and telco["TotalCharges"].isna().sum() == 0
print("fixed: TotalCharges ->", telco["TotalCharges"].dtype)

fixed: TotalCharges -> float64


### 1.3 Landmine #2 - inconsistent encodings

Two smaller issues, same root cause (this table was exported from several systems):

1. `SeniorCitizen` is `0/1` while every other flag is `Yes/No` - map it for consistency.
2. Six service columns use a third level, `"No internet service"` - but that fact is
   already fully encoded in `InternetService="No"`. Keeping it as a separate level
   triples encoder noise for zero information, so we collapse it to `"No"`.
   *(Trade-off, stated honestly: we lose the surface distinction; the information
   survives in `InternetService`.)*

Finally the target: `Churn` Yes/No = 1/0 integer.

In [29]:
service_cols = ["PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
                "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

telco["SeniorCitizen"] = telco["SeniorCitizen"].map({0: "No", 1: "Yes"})
telco[service_cols] = telco[service_cols].replace(
    {"No internet service": "No", "No phone service": "No"})
telco["Churn"] = (telco["Churn"] == "Yes").astype(int)

print("churn rate:", round(telco["Churn"].mean(), 4))
for c in ["SeniorCitizen", "MultipleLines", "OnlineSecurity"]:
    print(c, "->", telco[c].unique().tolist())

churn rate: 0.2654
SeniorCitizen -> ['No', 'Yes']
MultipleLines -> ['No', 'Yes']
OnlineSecurity -> ['No', 'Yes']


## 2. Online Retail II - load both sheets

Different animal entirely: a **transaction log** (one row per line item), not a
customer snapshot. Two things to know before reading:

- it is an xlsx with **two sheets** ("Year 2009-2010", "Year 2010-2011") that
  **overlap around December 2010** - concat then dedup;
- the read takes a couple of minutes (openpyxl parsing ~1M rows), which is exactly
  why the promoted loader caches to parquet.

In [30]:
import io, zipfile

with zipfile.ZipFile("../Dataset/online_retail_2.zip") as zf:
    data = io.BytesIO(zf.read("online_retail_II.xlsx"))
sheets = pd.read_excel(data, sheet_name=None)   # ~2 min on first run

for name, sheet in sheets.items():
    print(f"{name}: {sheet.shape}")
retail_raw = pd.concat(sheets.values(), ignore_index=True)
before = len(retail_raw)
retail_raw = retail_raw.drop_duplicates(ignore_index=True)
print(f"concat: {before:,} rows -> {len(retail_raw):,} after dedup "
      f"({before - len(retail_raw):,} exact duplicates from the sheet overlap)")
retail_raw.head()

Year 2009-2010: (525461, 8)
Year 2010-2011: (541910, 8)
concat: 1,067,371 rows -> 1,033,036 after dedup (34,335 exact duplicates from the sheet overlap)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### 2.1 Audit - three landmines at once

1. **Cancellations**: invoice numbers starting with `C` are credit notes (returns),
   with negative quantities.
2. **Missing `Customer ID`** on ~25% of rows - guest checkouts / data loss. Useless
   for *customer-level* analysis (RFM, CLV), so they are dropped for that purpose.
3. **Non-positive prices** - manual adjustments, bad rows.

In [31]:
retail_raw = retail_raw.rename(columns={"Customer ID": "CustomerID"})
retail_raw["Invoice"] = retail_raw["Invoice"].astype(str)
retail_raw["is_cancellation"] = retail_raw["Invoice"].str.startswith("C")

print(f"cancellations:      {retail_raw['is_cancellation'].mean():6.2%}")
print(f"missing CustomerID: {retail_raw['CustomerID'].isna().mean():6.2%}")
print(f"Quantity <= 0:      {(retail_raw['Quantity'] <= 0).mean():6.2%}")
print(f"Price <= 0:         {(retail_raw['Price'] <= 0).mean():6.2%}")
retail_raw[retail_raw["is_cancellation"]].head(3)

cancellations:       1.85%
missing CustomerID: 22.76%
Quantity <= 0:       2.18%
Price <= 0:          0.58%


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,is_cancellation
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,True
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,True
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,True


In [32]:
retail = retail_raw.loc[
    ~retail_raw["is_cancellation"]
    & retail_raw["CustomerID"].notna()
    & (retail_raw["Quantity"] > 0)
    & (retail_raw["Price"] > 0)
].copy()
retail["CustomerID"] = retail["CustomerID"].astype(int)
retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"])
retail["Revenue"] = retail["Quantity"] * retail["Price"]

print(f"clean purchases: {len(retail):,} rows, "
      f"{retail['CustomerID'].nunique():,} customers, "
      f"{retail['InvoiceDate'].min():%Y-%m-%d} -> {retail['InvoiceDate'].max():%Y-%m-%d}")

clean purchases: 779,425 rows, 5,878 customers, 2009-12-01 -> 2011-12-09


## 3. MegaFon uplift dataset

This one arrives via `scikit-uplift`'s fetcher (~100MB download). The promoted loader
caches it to `Dataset/megafon/megafon.parquet` on first fetch - run this cell once with
internet, every later run reads the cache in seconds.

Structure: 50 anonymized features `X_1...X_50`, a `treatment_group` flag (who got the
campaign) and a `conversion` outcome. The treatment column is what makes Lens C
(uplift) possible - and it is exactly what Telco lacks.

In [33]:
from src.data_loader import load_megafon, peek

mega = load_megafon()          # first call: downloads + caches; later: instant
peek(mega, "MegaFon")

=== MegaFon ===
shape: 600,000 rows x 52 columns
                   dtype  nulls  null_%  unique
X_1              float64      0     0.0  600000
X_2              float64      0     0.0  600000
X_3              float64      0     0.0  600000
X_4              float64      0     0.0  600000
X_5              float64      0     0.0  600000
X_6              float64      0     0.0  600000
X_7              float64      0     0.0  600000
X_8              float64      0     0.0  600000
X_9              float64      0     0.0  600000
X_10             float64      0     0.0  600000
X_11             float64      0     0.0  600000
X_12             float64      0     0.0  600000
X_13             float64      0     0.0  600000
X_14             float64      0     0.0  600000
X_15             float64      0     0.0  600000
X_16             float64      0     0.0  600000
X_17             float64      0     0.0  600000
X_18             float64      0     0.0  600000
X_19             float64      0     0.0

,X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,X_9,X_10,X_11,X_12,X_13,X_14,X_15,X_16,X_17,X_18,X_19,X_20,X_21,X_22,X_23,X_24,X_25,X_26,X_27,X_28,X_29,X_30,X_31,X_32,X_33,X_34,X_35,X_36,X_37,X_38,X_39,X_40,X_41,X_42,X_43,X_44,X_45,X_46,X_47,X_48,X_49,X_50,treatment_group,conversion
0,39.396577,-0.186548,19.524505,21.250208,55.291264,182.966712,-5.385606,144.573379,-12.534344,-58.279429,283.544750,-3.297557,74.491728,11.674656,-4.852843,99.324038,159.686346,-5.830570,-4.554391,-75.704888,-4.540720,107.891557,-0.592276,0.513937,-3.415607,-7.454581,-38.519996,-116.186909,18.253466,0.775597,-66.833169,121.204278,-58.588803,0.266334,0.758562,-13.254177,200.715580,77.227063,-230.591020,-3.023398,90.877638,134.363458,-213.584582,-2.092461,-93.973258,-0.155597,-312.130733,44.798182,-125.682413,16.231365,control,0
1,38.987694,0.819522,-42.064512,-48.270949,-33.171257,179.459341,-87.151810,-162.693257,20.651652,181.635081,114.382486,10.277246,132.852421,-14.088731,82.011396,-19.290410,-155.851825,3.904401,26.194543,87.908221,115.714068,-179.564296,-1.202563,1.827663,87.042091,6.771393,12.481973,304.514135,3.618649,-0.422986,76.520982,67.402922,-407.896510,-1.269765,-34.027547,-91.572702,61.215866,-80.652713,-782.791784,-33.541388,-183.840746,72.864779,559.783584,1.142391,80.037124,-1.216185,-111.473936,-127.737977,-117.501171,10.732234,control,0
2,-16.693093,1.844558,-8.615192,-18.818740,-22.271188,-116.290369,-63.816746,-38.340763,24.968496,-136.340629,628.310139,-7.714403,-4.863992,-41.638770,187.244122,16.716381,103.805977,15.248096,-7.963610,-18.581579,247.603883,-60.794763,-0.707689,-0.293637,186.251855,4.644561,-13.456976,52.765776,-95.096983,-2.191094,41.936533,17.929746,-243.263646,7.630227,-27.235726,38.757461,168.534929,94.979240,-177.641632,-40.559792,-203.637766,2.480242,96.998504,1.100962,-33.275159,0.920926,-679.492242,-91.009397,-18.173358,14.367636,treatment,0
3,-72.040154,-0.226921,39.802607,16.441262,-1.112509,68.128008,23.073147,4.688858,-49.383641,-91.866107,374.208539,-5.197953,-16.972788,-20.513819,26.733826,-5.051170,266.620852,-1.003867,5.644895,101.563496,24.222458,15.445679,-0.193439,-0.537621,18.220545,-19.826745,-9.969067,-176.056848,-34.994979,-0.169379,20.194249,97.909989,41.396933,-4.572652,73.594603,59.882086,-18.703877,109.139592,-4.272029,18.375161,172.906875,83.951551,-323.642557,-0.369182,93.221948,-1.962380,-442.466684,-22.298302,-75.916603,11.634299,treatment,0
4,18.296973,0.996437,24.465307,-34.151971,24.623458,-155.455558,-12.159787,26.705778,105.864805,258.607252,-555.119130,14.632472,100.997894,-10.130204,52.278128,-6.223072,-91.566718,24.962987,16.902873,-125.359051,77.880711,65.763350,-0.250273,0.470302,58.583197,24.301061,7.895425,64.681136,0.208348,1.215403,-32.063615,1.182104,-15.530264,9.247841,-40.267539,27.330542,-78.737771,-79.762654,-101.123230,-30.324588,125.577535,-208.531112,118.902324,-0.808578,-117.497906,1.770635,627.395611,122.019189,194.091195,-11.883858,treatment,0


In [34]:
mega.groupby("treatment_group")["conversion"].agg(n="size", conversion_rate="mean")

,n,conversion_rate
treatment_group,,
control,299632,0.179400
treatment,300368,0.228919


## 4. Promoted to `src/`

Everything above proved out, so it now lives in **`src/data_loader.py`**:


In [ ]:
from src.data_loader import load_telco, load_retail, save_processed

telco_v = load_telco()
retail_v = load_retail()        # second run: served from the parquet cache

assert telco_v.shape == telco.shape and len(retail_v) == len(retail)
save_processed(telco_v, "telco_clean")
save_processed(retail_v, "retail_clean")
print("saved: Dataset/processed/telco_clean.parquset, retail_clean.parquet")

saved: Dataset/processed/telco_clean.parquet, retail_clean.parquet


---
**Next:** `02_eda_visualization.ipynb` - what actually drives churn in each dataset,
using these loaders as one-liners.